In [1]:
import pandas as pd

In [5]:
fastaid_annot_id = pd.read_csv('../data/ssn/input_data/loading_SSN_annot.csv')
fastaid_annot_id['id'] = fastaid_annot_id['id'].astype(str)

clusts96 = pd.read_csv('../data/ssn/input_data/mmseqs_clustering/gh18domIPscan_mmseqs_clusters_96.0.csv').rename(columns={'cluster': 'cluster96'})
clust293 = pd.read_csv('../data/ssn/input_data/mmseqs_clustering/gh18domIPscan_mmseqs_clusters_293.0.csv').rename(columns={'cluster': 'cluster293'})
clust162 = pd.read_csv('../data/ssn/input_data/mmseqs_clustering/gh18domIPscan_mmseqs_clusters_162.0.csv').rename(columns={'cluster': 'cluster162'})
clust473 = pd.read_csv('../data/ssn/input_data/mmseqs_clustering/gh18domIPscan_mmseqs_clusters_473.0.csv').rename(columns={'cluster': 'cluster473'})

#merge all cluster dataframes and then merge with fastaid_annot_id
clusts = clusts96.merge(clust293, how='left', on='id').merge(clust162, how='left', on='id').merge(clust473, how='left', on='id')
#ensure all id columns are strings
clusts['id'] = clusts['id'].astype(str)

fastaid_annot_id_clusts = fastaid_annot_id.merge(clusts, how='left', left_on='id', right_on='id')
#ensure all values in cluster columns are strings
fastaid_annot_id_clusts['cluster96'] = fastaid_annot_id_clusts['cluster96'].astype(str)
fastaid_annot_id_clusts['cluster293'] = fastaid_annot_id_clusts['cluster293'].astype(str)
fastaid_annot_id_clusts['cluster162'] = fastaid_annot_id_clusts['cluster162'].astype(str)
fastaid_annot_id_clusts['cluster473'] = fastaid_annot_id_clusts['cluster473'].astype(str)
fastaid_annot_id_clusts

/var/folders/fg/5tjhqq3n0t70943jzd8k47800000gp/T/ipykernel_90271/704695833.py:1: DtypeWarning: Columns (40: Organism (ID)) have mixed types. Specify dtype option on import or set low_memory=False.
  fastaid_annot_id = pd.read_csv('../data/ssn/input_data/loading_SSN_annot.csv')


,Unnamed: 0,protein_accession,sequence_md5,sequence_length,analysis,signature_accession,signature_description,start,end,score,...,family,genus,species,Taxid,id,header,cluster96,cluster293,cluster162,cluster473
0,0,A0A010YG22,BCAC65E13E44ACC0D5889A40D5B1A2F8,354,PROSITE profiles,PS51910,Glycosyl hydrolases family 18 (GH18) domain pr...,56,354,1.020352e+01,...,Cryptosporangiaceae,Cryptosporangium,Cryptosporangium arvum,927661.0,0.0,A0A010YG22|1of1,NaN,NaN,NaN,NaN
1,1,A0A011PM39,2B03A5B39A2C5137BCACBC4D01697D06,1156,Pfam,PF00704,Glycosyl hydrolases family 18,137,428,2.300000e-16,...,NaN,Candidatus Accumulibacter,NaN,522306.0,NaN,NaN,NaN,NaN,NaN,NaN
2,2,A0A011PM39,2B03A5B39A2C5137BCACBC4D01697D06,1156,CATH-Gene3D,G3DSA:3.20.20.370,Glycoside hydrolase/deacetylase,503,716,1.200000e-57,...,NaN,Candidatus Accumulibacter,NaN,522306.0,NaN,NaN,NaN,NaN,NaN,NaN
3,3,A0A011PXQ4,BCAE3BD21D067344C874DD0EB6344E16,1156,Pfam,PF00704,Glycosyl hydrolases family 18,136,428,2.200000e-16,...,NaN,Candidatus Accumulibacter,NaN,1454003.0,NaN,NaN,NaN,NaN,NaN,NaN
4,4,A0A011PXQ4,BCAE3BD21D067344C874DD0EB6344E16,1156,CATH-Gene3D,G3DSA:3.20.20.370,Glycoside hydrolase/deacetylase,504,716,3.400000e-57,...,NaN,Candidatus Accumulibacter,NaN,1454003.0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41899,41899,X6Q4Q0,83C4D5DBFD8A1179D0BA7A5552A33164,1132,CATH-Gene3D,G3DSA:3.20.20.370,Glycoside hydrolase/deacetylase,488,704,6.800000e-55,...,Prevotellaceae,Prevotella,NaN,1161412.0,NaN,NaN,NaN,NaN,NaN,NaN
41900,41900,X8GPF2,70D3A80C4982E83860FD2DA40F7823CA,567,Pfam,PF00704,Glycosyl hydrolases family 18,334,554,2.100000e-20,...,Lachnospiraceae,Shuttleworthella,NaN,936574.0,38200.0,X8GPF2|1of1,NaN,NaN,NaN,NaN
41901,41901,X8HE75,A456F3A5EAF449088DBE76505D69FC5A,539,Pfam,PF00704,Glycosyl hydrolases family 18,206,422,2.900000e-19,...,Veillonellaceae,Veillonella,NaN,936591.0,38201.0,X8HE75|1of1,NaN,NaN,NaN,NaN
41902,41902,Z9JN00,A28869D690FF03DAD30FDC1D4D83035F,351,Pfam,PF00704,Glycosyl hydrolases family 18,37,336,1.000000e-21,...,Lysobacteraceae,Xylella,NaN,1444770.0,38202.0,Z9JN00|1of1,NaN,NaN,NaN,NaN


In [6]:
def simpsons_diversity(signature_series):
    """Unbiased (finite-sample) Simpson's Diversity Index, scaled 0-100.
    0 = all sequences share one signature. 100 = maximal heterogeneity."""
    counts = signature_series.value_counts()
    N = counts.sum()
    if N <= 1:
        return float('nan')  # undefined for singleton clusters
    D = 1 - sum(n * (n - 1) for n in counts) / (N * (N - 1))
    return D * 100

def compute_cluster_heterogeneity(df, cluster_col, id_col='header',
                                    signature_col='signature_accession_fingerprint_GH18replaced'):
    seq_level = df.dropna(subset=[cluster_col, id_col]).drop_duplicates(subset=[id_col])

    grouped = seq_level.groupby(cluster_col).agg(
        numuniqsignatures=(signature_col, 'nunique'),
        totalnumseqs=(id_col, 'nunique')
    ).reset_index()
    grouped = grouped.rename(columns={cluster_col: 'clusterid'})

    simpson = seq_level.groupby(cluster_col)[signature_col].apply(simpsons_diversity)
    grouped['heterogeneity_simpson'] = grouped['clusterid'].map(simpson)

    return grouped

cluster_cols = ['cluster96', 'cluster162', 'cluster293', 'cluster473']
cluster_results = {col: compute_cluster_heterogeneity(fastaid_annot_id_clusts, col) for col in cluster_cols}

In [30]:
import os

outdir = '../output/cluster_heterogeneity'
os.makedirs(outdir, exist_ok=True)

for col, result_df in cluster_results.items():
    result_df.to_csv(f'{outdir}/{col}_heterogeneity.csv', index=False)